# Семинар: Рекуррентные архитектуры, Fine-Tuning и Дистилляция знаний

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import nltk
import numpy as np
import time
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm
import copy

# Проверка доступности GPU и настройка устройства
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Доступные устройства: CUDA={torch.cuda.is_available()}")
print(f"Используемое устройство: {device}")
print(f"Версия PyTorch: {torch.__version__}")

# Скачивание ресурсов NLTK (выполняется один раз)
nltk.download('punkt_tab', quiet=True)

### Подготовка данных и токенизация

![alt](../data/dataset-cover.jpg)

Датасет IMDB содержит 50 000 рецензий с бинарными метками (положительный/отрицательный). Для воспроизводимости и ускорения обучения в рамках семинара будем использовать сбалансированную подвыборку. Токенизация выполняется через `nltk.word_tokenize`, что позволяет разбивать текст на слова, приводить к нижнему регистру и отделять пунктуацию.

In [ ]:
# Загрузка датасета
dataset = load_dataset("imdb")
dataset['train'] = dataset['train'].shuffle(seed=42)

In [ ]:
# train и test
train_texts = dataset['train']['text'][:10000]
train_labels = dataset['train']['label'][:10000]
test_texts = dataset['test']['text'][:5000]
test_labels = dataset['test']['label'][:5000]

# valid
valid_texts = dataset['train']['text'][10000:15000]
valid_labels = dataset['train']['label'][10000:15000]

In [ ]:
def tokenize_text(text):
    return nltk.word_tokenize(text.lower())

In [ ]:
# Построение словаря (только на train)
token_counter = Counter()
for t in train_texts:
    token_counter.update(tokenize_text(t))

vocab = ['<pad>', '<unk>'] + [token for token, _ in token_counter.most_common(8000)]
vocab_size = len(vocab)
word2idx = {word: i for i, word in enumerate(vocab)}

In [ ]:
word2idx['the']

In [ ]:
MAX_LEN = 128

def encode_texts(texts):
    encoded = []
    for t in texts:
        tokens = tokenize_text(t)[:MAX_LEN]
        ids = [word2idx.get(tok, word2idx['<unk>']) for tok in tokens]
        ids += [word2idx['<pad>']] * (MAX_LEN - len(ids))
        encoded.append(ids)
    return torch.tensor(encoded, dtype=torch.long)

train_x = encode_texts(train_texts)
train_y = torch.tensor(train_labels, dtype=torch.long)

valid_x = encode_texts(valid_texts)
valid_y = torch.tensor(valid_labels, dtype=torch.long)

test_x = encode_texts(test_texts)
test_y = torch.tensor(test_labels, dtype=torch.long)

In [ ]:
class TextDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [ ]:
train_loader = DataLoader(TextDataset(train_x, train_y), batch_size=32, shuffle=True)
valid_loader = DataLoader(TextDataset(valid_x, valid_y), batch_size=64)
test_loader = DataLoader(TextDataset(test_x, test_y), batch_size=64)

## Проблемы RNN

> Примечание: Операция $\odot$ обозначает поэлементное матричное умножение

Проблема стандартных RNN заключается в затухании или взрыве градиентов. Почему возникает такая проблема? Можно узнать подробно [тут](https://education.yandex.ru/handbook/ml/article/nejroseti-dlya-raboty-s-posledovatelnostyami#rekurrentnye-nejronnye-seti) 

Если кратко:
На лекции мы выяснили, что выход $y_n$ зависит от входа $x_n$ и скрытого состояния $h_n$, который в свою очередь зависит от всех предыдущих скрытых состояний $h_i = tanh(h_{i-1}W_1 + x_iW_2$.

Если мы теперь посчитаем градиент при backprop:

$$\nabla_{h_{i-1}}L  = (\nabla_{h_{i-1}}L)W_1^T \odot tanh'(h_{i-1}W1 +x_iW2)$$


Если собственные значения $\lambda$ матрицы $W_1^T$ превышают 1, то градиент взрывается, если меньше 1, то градиент затухает.

## LSTM

Сеть с долговременной и кратковременной памятью (Long short term memory, LSTM) частично решает проблему исчезновения или зашкаливания градиентов в процессе обучения рекуррентных сетей методом обратного распространения ошибки. 

![alt](../data/lstm.png)

Все рекуррентные сети можно представить в виде цепочки из повторяющихся блоков. В RNN таким блоком обычно является один линейный слой с гиперболическим тангенсом в качестве функции активации. В LSTM повторяющийся блок имеет более сложную структуру, состоящую не из одного, а из четырех слоев. Кроме скрытого состояния $h_n$ в LSTM вводится понятие блока \ ячейки (`cell state`, c_n)

Этот `cell state` будет играть роль внутренней, закрытой информации LSTM-блока.

LSTM может добавлять или удалять определенную информацию из cell state с помощью специальных механизмов, которые называются `gates`

Рассмотрим этот механизм подробнее.

Основное назначение `gate` — контролировать количество проходящей через него информации. Для этого матрица, проходящая по каналу, который контролирует вентиль, поточечно умножается на выражение вида

$$\sigma(W_1h_{n-1} + W_2x_n)$$

По сути, определяется коэф. пропускной способности (`capacity`) от 0 до 1

### Forget gate

![alt](../data/forget.webp)

Forget gate (вентиль забывания). Он позволяет на основе предыдущего скрытого состояния $h_{t-1}$ и нового входа $x_t$ определить, какую долю информации из $c_{t-1}$ стоит пропустить дальше, а какую забыть.

доля $f_t$ сохраняемой информации из $c_{t-1}$ вычисляется так:

$$f_t = \sigma(g_{t-1}W_1 +x_tW_2 + b)$$

### Input gate

![alt](../data/input.webp)

Вычислим 

$$\tilde{C} = tanh(h_{t-1}W_1 + x_tW_2 + b)$$

и

$$i_t = \sigma(h_{t-1}W_1 + x_tW_2 + b)$$

Обновим `cell state`

![alt](../data/cell.webp)

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{C_t}$$


### Output gate

Он отвечает на вопрос о том, сколько информации из `cell state` следует отдавать на выход из LSTM-блока. Доля вычисляется следующим образом:

$$o_t = \sigma(h_{t-1}W_1 + x_tW_2 + b)$$

Теперь пропускаем cell state через гиперболический тангенс, чтобы значения были в диапазоне от $-1$ до $1$
1, и умножаем полученный вектор на $o_n$, чтобы отфильтровать информацию из `cell state`, которую нужно подать на выход:

$$h_t = o_t \odot tanh(c_t)$$

![alt](../data/output.webp)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True,
                           bidirectional=True, dropout=0.3, num_layers=2)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.hidden_dim = hidden_dim

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = self.dropout(x)
        x = torch.cat((x[:, -1, :self.hidden_dim], x[:, 0, self.hidden_dim:]), dim=1)
        return self.fc(x)

In [ ]:
# Вспомогательная функция для вычисления метрик
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)  # Один вызов
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

In [ ]:
def train_rnn_model(model, train_loader, valid_loader, test_loader, epochs=5, lr=1e-3, patience=3):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    best_val_loss = float('inf')
    patience_counter = 0
    best_model = None

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, valid_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1} | Train Acc: {train_acc:.3f} | Val Acc: {val_acc:.3f} | Val Loss: {val_loss:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    if best_model is not None:
        model.load_state_dict(best_model)

    # Финальное тестирование
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(f"\nTest Accuracy: {test_acc:.3f} | Test Loss: {test_loss:.4f}")
    history['test_acc'] = test_acc

    return history

In [ ]:
lstm_model = LSTMClassifier(vocab_size=vocab_size, embed_dim=256, hidden_dim=256, num_classes=2)
h_lstm = train_rnn_model(lstm_model, train_loader, valid_loader, test_loader, epochs=10)

Описанная архитектура выглядит несколько сложно. Кроме того, вычисление четырех различных типов гейтов может быть вычислительно невыгодным. Поэтому были разработаны различные вариации LSTM, одна из самых популярных (Gated Recurrent Unit, GRU) освещена ниже.

Gated Recurrent Unit был предложен в [статье](https://arxiv.org/pdf/1406.1078v3.pdf) Cho et al. в 2014 году. GRU объединяет input gate и forget gate в один update gate, также устраняет разделение внутренней информации блока на hidden и cell state. Вот общий вид GRU-блока:

![alt](../data/gru.webp)

Внимательно посмотрев на структуру LSTM, можно заметить, что функции forget gate и input gate похожи. Первый механизм определяет, какие значения $c_{t-1}$ надо забыть, а второй — какие значения нового вектора $\tilde{C}$ нужно использовать для обновления старого `cell state` $c_{t-1}$ Давайте объединим эти функции воедино: грубо говоря, будем забывать только те значения, которые собираемся обновить. Такую роль выполняет `update gate`

$$u_t = \sigma(h_{t-1}W_1^u + x_tW_2^u + b)$$

Ещё один новый тип гейта, который появляется в GRU — `reset gate`. Он определяет, какую долю информации из $h_{t-1}$ с прошлого шага надо «сбросить», инициализировать заново.

$$r_t = \sigma(h_{t-1}W_1^r + x_tW_2^r + b)$$

Теперь мы вычисляем потенциальное обновление для скрытого состояния

$$\tilde{h} = tanh((r_t \odot h_{t-1})W_1^h + x_tW_2^h + b)$$

и, наконец, решаем, что из старого забыть, а что из нового добавить:

$$h_t = (1-z_t) \odot h_{t-1} + z_t \odot \tilde{h}$$


В итоге GRU имеет меньше параметров, чем LSTM (в GRU нет output gate) и при прочих равных, быстрее учится. GRU и LSTM показывают сопоставимое качество на многих задачах, включая генерацию музыки, распознавание речи, многие задачи обработки естественного языка.

Реализация аналогична LSTM с заменой `nn.LSTM` на `nn.GRU`.

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers = 2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3 if num_layers > 1 else 0,
            bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.hidden_dim = hidden_dim

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.gru(x)
        x = self.dropout(x)
        x = torch.cat((x[:, -1, :self.hidden_dim], x[:, 0, self.hidden_dim:]), dim=1)
        x = self.fc(x)
        return x

gru_model = GRUClassifier(vocab_size=vocab_size, embed_dim=256, hidden_dim=256, num_classes=2)
h_gru = train_rnn_model(gru_model, train_loader, valid_loader, test_loader, epochs=10)

## Fine-Tuning и Дистилляция знаний

1. **Fine-Tuning:** Процесс адаптации предобученной модели к новой задаче. В отличие от обучения с нуля, мы берём веса, полученные на больших корпусах текстов, и дообучаем последние слои (или всю модель) на целевом датасете. 

![alt](../data/fine.webp)



2. **Knowledge Distillation (KD):** Метод сжатия моделей, при котором большая "учитель" обучает компактного "студента". Студент минимизирует не только кросс-энтропию с истинными метками, но и KL-дивергенцию между своими логитами и логитами учителя.


![alt](../data/dis.webp)



`DistilBERT` Модель - студент, полученная дистилляцией из BERT-base. Сохраняет ~97% качества BERT при сокращении параметров на 40% и ускорении инференса на 60%. 

Сейчас посмотрим, как можно сделать Fine-Tuning уже дистиллированной модели на датасете IMDB.

### BERT


[BERT](https://huggingface.co/papers/1810.04805) — это двунаправленный трансформер, предварительно обученный на немаркированном тексте для прогнозирования замаскированных токенов в предложении и определения того, следует ли одно предложение за другим. Основная идея заключается в том, что, случайным образом маскируя некоторые токены, модель может обучаться на тексте слева и справа, что позволяет ей лучше понимать контекст. Кроме того, BERT очень универсален, поскольку изученные языковые представления можно адаптировать для решения других задач в области обработки естественного языка, дообучив дополнительный слой или блок.


![alt](../data/bert.png)

BERT - это настоящий трансформер. А про них мы поговорим в следующий раз.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

dbert_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
class DistilBERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        )
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}, self.labels[idx]

In [ ]:
dbert_train_loader = DataLoader(
    DistilBERTDataset(train_texts, train_labels, dbert_tokenizer),
    batch_size=32,
    shuffle=True
)
dbert_valid_loader = DataLoader(
    DistilBERTDataset(valid_texts, valid_labels, dbert_tokenizer),
    batch_size=32
)
dbert_test_loader = DataLoader(
    DistilBERTDataset(test_texts, test_labels, dbert_tokenizer),
    batch_size=32
)

In [ ]:
# from transformers import DistilBertForSequenceClassification, DistilBertConfig

# config = DistilBertConfig.from_pretrained(
#     "distilbert-base-uncased",
#     dropout=0.3,
#     attention_dropout=0.3,
#     seq_classif_dropout=0.3
# )


model_db = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
model_db.to(device)

In [ ]:
def train_distilbert(model, train_loader, valid_loader, test_loader, epochs=5, lr=2e-5, patience=2):
    """
    Обучение DistilBERT с правильной регуляризацией и валидацией
    """
    # Используем AdamW с weight decay (стандарт для трансформеров)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    # Learning rate scheduler (важно для тонкой настройки)
    total_steps = len(train_loader) * epochs
    scheduler = optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1.0,
        end_factor=0.0,
        total_iters=total_steps
    )

    criterion = nn.CrossEntropyLoss()
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'test_loss': [], 'test_acc': []
    }

    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss, train_acc = 0.0, 0.0
        total_train = 0

        for batch, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            # Move to device
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = labels.to(device)

            # Forward pass
            optimizer.zero_grad()
            outputs = model(**batch).logits
            loss = criterion(outputs, labels)

            # Backward pass с gradient clipping
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            # Metrics
            train_loss += loss.item() * labels.size(0)
            _, preds = torch.max(outputs, 1)
            train_acc += (preds == labels).sum().item()
            total_train += labels.size(0)

        train_loss /= total_train
        train_acc /= total_train

        # Validation phase
        val_loss, val_acc = evaluate_distilbert(model, valid_loader, criterion, device)

        # Save metrics
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
        print(f"LR: {optimizer.param_groups[0]['lr']:.2e}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
            print(f"  ✓ New best model saved!")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  ✗ Early stopping triggered!")
                break

    # Load best model and evaluate on test set
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_loss, test_acc = evaluate_distilbert(model, test_loader, criterion, device)
    history['test_loss'] = test_loss
    history['test_acc'] = test_acc

    print(f"\n{'='*50}")
    print(f"TEST RESULTS:")
    print(f"Test Accuracy: {test_acc:.4f} | Test Loss: {test_loss:.4f}")
    print(f"{'='*50}")

    return history, model

def evaluate_distilbert(model, loader, criterion, device):
    """Функция оценки для DistilBERT"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch, labels in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = labels.to(device)

            outputs = model(**batch).logits
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
history_db, model_db_trained = train_distilbert(
    model_db,
    dbert_train_loader,
    dbert_valid_loader,  # Теперь передаем validation, не test!
    dbert_test_loader,
    epochs=5,
    lr=2e-5,
    patience=2
)

### Метрики и время инференса

`Инференс` - это этап **использования уже обученной модели** для получения предсказаний **на новых данных**.

`Время инференса` — это время, которое требуется модели для обработки одного запроса (или батча) и получения предсказания.

В продакшене критично время инференса. Для корректного измерения на GPU необходимо использовать `torch.cuda.synchronize()`, так как операции выполняются асинхронно. Сравним среднее время обработки батча для LSTM, GRU и DistilBERT.

In [ ]:
def measure_inference_time(model, data_loader, is_transformer=False, model_name="Model"):
    model.eval()
    times = []
    with torch.no_grad():
        for batch in data_loader:
            if is_transformer:
                inputs = {k: v.to(device) for k, v in batch[0].items()}
            else:
                inputs, _ = batch
                inputs = inputs.to(device)

            torch.cuda.synchronize()
            start = time.perf_counter()
            _ = model(**inputs) if is_transformer else model(inputs)
            torch.cuda.synchronize()
            end = time.perf_counter()
            times.append(end - start)

    avg_time = np.mean(times)
    print(f"{model_name} | Avg Inference Time per Batch: {avg_time:.4f} sec")
    return avg_time

print("\n--- Измерение времени инференса ---")
t_lstm = measure_inference_time(lstm_model, test_loader, model_name="LSTM")
t_gru = measure_inference_time(gru_model, test_loader, model_name="GRU")
t_dbert = measure_inference_time(model_db, dbert_test_loader, is_transformer=True, model_name="DistilBERT")

### Визуализация Loss и Accuracy
Сравним динамику обучения LSTM и DistilBERT. Обратите внимание, как быстро сходятся предобученные модели по сравнению с архитектурой, обучаемой с нуля.

In [ ]:
# Компактное сравнение: Validation Loss + Test Accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Validation Loss
ax1.plot(range(1, len(h_lstm['val_loss']) + 1), h_lstm['val_loss'],
         'b-o', label='LSTM', linewidth=2, markersize=6)
ax1.plot(range(1, len(history_db['val_loss']) + 1), history_db['val_loss'],
         'r-o', label='DistilBERT', linewidth=2, markersize=6)
ax1.set_title('Validation Loss', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Test Accuracy
models = ['LSTM', 'DistilBERT']
accuracies = [h_lstm.get('test_acc', 0), history_db.get('test_acc', 0)]
colors = ['blue', 'red']
bars = ax2.bar(models, accuracies, color=colors, alpha=0.7)
ax2.set_title('Test Accuracy', fontsize=12, fontweight='bold')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1.05)
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# Добавляем значения на столбцы
for bar, acc in zip(bars, accuracies):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Вывод результатов
print(f"\nValidation Loss - LSTM: {min_lstm_val_loss:.4f}, DistilBERT: {min_db_val_loss:.4f}")
print(f"Test Accuracy - LSTM: {h_lstm.get('test_acc', 0):.4f}, DistilBERT: {history_db.get('test_acc', 0):.4f}")

### Интерактивный блок
Введите произвольный отзыв на английском языке. Код выполнит предобработку, передаст данные через все три модели и выведет вероятностные предсказания.

In [ ]:
def predict_review_improved(text, show_details=True):
    """
    Улучшенная функция предсказания с дополнительной информацией
    """
    # 1. Добавим очистку текста
    text = text.strip()
    if not text:
        print("Ошибка: пустой текст")
        return

    # 2. Добавим измерение времени инференса
    start_time = time.time()

    # 3. Обработка для LSTM/GRU с обработкой ошибок
    try:
        tokens = tokenize_text(text)[:MAX_LEN]
        ids = [word2idx.get(tok, word2idx['<unk>']) for tok in tokens]
        ids += [word2idx['<pad>']] * (MAX_LEN - len(ids))
        inp_rnn = torch.tensor([ids], dtype=torch.long).to(device)
    except Exception as e:
        print(f"Ошибка токенизации для RNN: {e}")
        return

    # 4. Обработка для DistilBERT
    try:
        inp_db = dbert_tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN)
        inp_db = {k: v.to(device) for k, v in inp_db.items()}
    except Exception as e:
        print(f"Ошибка токенизации для DistilBERT: {e}")
        return

    # 5. Инференс
    with torch.no_grad():
        lstm_logits = lstm_model(inp_rnn)
        gru_logits = gru_model(inp_rnn)
        db_logits = model_db(**inp_db).logits

        # Softmax с числовой стабильностью
        lstm_probs = torch.softmax(lstm_logits, dim=1).cpu().numpy()[0]
        gru_probs = torch.softmax(gru_logits, dim=1).cpu().numpy()[0]
        db_probs = torch.softmax(db_logits, dim=1).cpu().numpy()[0]

    inference_time = time.time() - start_time

    # 6. Результаты
    label_map = {0: "Negative", 1: "Positive"}

    print(f"\n{'='*60}")
    print(f"ТЕКСТ: {text[:100]}{'...' if len(text) > 100 else ''}")
    print(f"{'='*60}")

    results = [
        ("LSTM", lstm_probs, 'blue'),
        ("GRU", gru_probs, 'green'),
        ("DistilBERT", db_probs, 'red')
    ]

    for name, probs, color in results:
        pred_class = np.argmax(probs)
        confidence = probs.max()
        sentiment = label_map[pred_class]

        # Эмодзи для наглядности
        emoji = "👍" if sentiment == "Positive" else "👎"

        print(f"{name:12} {emoji} {sentiment:8} (Confidence: {confidence:.3f})")

        if show_details:
            # Показываем распределение вероятностей
            print(f"             Probabilities: Negative: {probs[0]:.3f}, Positive: {probs[1]:.3f}")
    print(f"{'='*60}\n")

    return {
        'text': text,
        'lstm': {'sentiment': label_map[np.argmax(lstm_probs)], 'confidence': lstm_probs.max()},
        'gru': {'sentiment': label_map[np.argmax(gru_probs)], 'confidence': gru_probs.max()},
        'distilbert': {'sentiment': label_map[np.argmax(db_probs)], 'confidence': db_probs.max()},
        'inference_time': inference_time
    }

result = predict_review_improved(
   # "The cinematography was stunning, but the plot was incredibly predictable and boring.",
    "This movie is absolutely amazing! Best film I've seen all year.",
    show_details=True
)